# Notebook 3: Instruction Tuning (Stage 2, optional)

**Runtime: L4 / T4.**

## Purpose
The **local, opt-in** stage. Each org takes a shared DAPT adapter from notebook 2 and
tunes it on **its own** instruction data — no aggregation, data never leaves.
This notebook is entirely skippable: an org can deploy the DAPT-only model from
notebook 2 and never run this. We keep it optional *and* evaluate both paths, so the
results table has "with instruction tuning" vs "without" rows.

## Warm-starting
Stage 2 continues the **same** LoRA adapter from its DAPT state into an
instruction-tuned state. `INIT_FROM = None` gives the "instruction-only, no DAPT"
ablation; a DAPT adapter id gives "DAPT → instruction".

## Artifact contract
```
adapters/<dapt_id>/            # INPUT  — a Stage-1 adapter (or None)
corpus/instruction_clients_v2/ # INPUT  — labeled pairs per client
corpus/eval/heldout_ids.json   # INPUT  — excluded from training
adapters/<dapt_id>__instr__<client>/   # OUTPUT — per-org instruction adapter + meta
```

In [ ]:
!pip -q install -U "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" \
  "peft>=0.10.0" "bitsandbytes>=0.43.0" sentencepiece "pandas==2.2.2"

In [ ]:
import os, json, math, random
from datetime import datetime
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, PeftModel

import os, tempfile
def _load_dotenv(path='.env'):
    # Minimal .env loader (no dependency); real environment vars take precedence.
    try:
        with open(path) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#') or '=' not in _line:
                    continue
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))
    except FileNotFoundError:
        pass
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
_load_dotenv()   # pull NVD_API_KEY / FEDDAPT_ROOT from .env if present
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', '/content/drive/MyDrive/FedDAPT')
else:
    # PyCharm / remote GPU: set FEDDAPT_ROOT to your data dir, e.g.  export FEDDAPT_ROOT=/data/fedapt
    PROJECT_ROOT = os.environ.get('FEDDAPT_ROOT', os.path.abspath('./FedDAPT'))
SCRATCH = os.environ.get('FEDDAPT_WORK', os.path.join(tempfile.gettempdir(), 'fedapt_work'))
os.makedirs(PROJECT_ROOT, exist_ok=True); os.makedirs(SCRATCH, exist_ok=True)
print('PROJECT_ROOT =', PROJECT_ROOT, '| SCRATCH =', SCRATCH)
INSTR_DIR   = f'{PROJECT_ROOT}/corpus/instruction_clients_v2'
EVAL_DIR    = f'{PROJECT_ROOT}/corpus/eval'
ADAPTER_DIR = f'{PROJECT_ROOT}/adapters'
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device =', device)

In [ ]:
MODEL_ID = 'mistralai/Mistral-7B-v0.1'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=['q_proj','v_proj'],
                         lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
CONFIG = {'learning_rate': 1e-5,          # lower: we continue an already-adapted adapter
          'batch_size': 1, 'grad_accum': 8, 'max_seq_length': 512, 'epochs': 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

def build_model(init_from):
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    if init_from:                                  # warm-start from a Stage-1 DAPT adapter
        return PeftModel.from_pretrained(base, f'{ADAPTER_DIR}/{init_from}', is_trainable=True)
    return get_peft_model(base, lora_config)       # None => instruction-only ablation

def tok_fn(b):
    out = tokenizer(b['text'], truncation=True, max_length=CONFIG['max_seq_length'], padding='max_length')
    out['labels'] = out['input_ids'].copy(); return out

def doc_key(t):
    import hashlib; return hashlib.md5(t.encode('utf-8')).hexdigest()[:12]
print('ready')

## 1. Choose what to do (the opt-in switches)
`RUN = False` here skips the whole stage. `INIT_SOURCES` lists which starting
points to instruction-tune from — include `None` to also produce the no-DAPT
ablation in the same run.

In [ ]:
RUN = True
INIT_SOURCES = [
    None,                    # A: instruction-only (no DAPT)
    'dapt_fedavg_no_dp',     # C: federated DAPT -> instruction (headline)
    'LOCAL',                 # B: each client uses its OWN local DAPT adapter (notebook 2 §2b)
]
client_names = ['client_a_endpoint', 'client_b_network', 'client_c_cloud']

heldout = set(json.load(open(f'{EVAL_DIR}/heldout_ids.json'))) if os.path.exists(f'{EVAL_DIR}/heldout_ids.json') else set()
print('held-out excluded:', len(heldout))

instr_texts = {}
for n in client_names:
    docs = json.load(open(f'{INSTR_DIR}/{n}.json'))
    instr_texts[n] = [d for d in docs if doc_key(d) not in heldout]
    print(f'{n}: {len(instr_texts[n])} instruction docs')

In [ ]:
def instruction_tune_one(init_from, client, texts):
    out_id = f'{init_from or "nodapt"}__instr__{client}'
    out_path = f'{ADAPTER_DIR}/{out_id}'
    if os.path.exists(f'{out_path}/meta.json'):
        print('skip (done):', out_id); return
    model = build_model(init_from)
    ds = Dataset.from_list([{'text': t} for t in texts]).map(tok_fn, batched=True, remove_columns=['text'])
    Trainer(model=model,
        args=TrainingArguments(output_dir=f'{SCRATCH}/{out_id}',
            per_device_train_batch_size=CONFIG['batch_size'], gradient_accumulation_steps=CONFIG['grad_accum'],
            num_train_epochs=CONFIG['epochs'], learning_rate=CONFIG['learning_rate'],
            bf16=torch.cuda.is_available(), optim='paged_adamw_8bit',
            save_strategy='no', report_to='none', logging_steps=20),
        train_dataset=ds, data_collator=collator).train()
    model.save_pretrained(out_path)
    json.dump({'id': out_id, 'stage': 'instruction', 'method': 'local',
               'init_from': init_from, 'client': client, 'adapter_path': out_path,
               'epsilon': 'inf', 'mu': 0.0, 'created': datetime.now().isoformat()},
              open(f'{out_path}/meta.json', 'w'), indent=2)
    del model; torch.cuda.empty_cache(); print('saved:', out_id)

if RUN:
    for src in INIT_SOURCES:
        for c in client_names:
            # 'LOCAL' => each org warm-starts from its OWN local DAPT adapter (ablation B)
            actual = f'dapt_local_{c}' if src == 'LOCAL' else src
            instruction_tune_one(actual, c, instr_texts[c])
    print('instruction tuning complete')
else:
    print('RUN=False — Stage 2 skipped (DAPT-only deployment).')

## Next
**notebook 4** evaluates every adapter it finds — the DAPT-only ones from notebook 2 and the
instruction-tuned ones from here — so "with vs without instruction tuning" falls
straight out of the results table.